# Vionex #DEEPX — Arabic ABSA Dataset EDA (Baseline)

This notebook is designed to run on **Kaggle Notebooks**.

Goals:
- Simple EDA: schema, missingness, label balance, text length.
- Semantic EDA (lightweight): TF-IDF → SVD 2D visualization + KMeans clusters + top terms.
- (Optional) Weak aspect labeling using the repo script `absa_aspect_labeling.py`.


## 0) Setup

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8')
pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_columns', 200)


## 1) Load dataset

### Option A: KaggleHub (if your environment supports it)
If KaggleHub isn't available on Kaggle runtime, use **Option B**.


In [ ]:
# If needed:
# !pip -q install kagglehub[pandas-datasets]

USE_KAGGLEHUB = False

df = None
if USE_KAGGLEHUB:
    import kagglehub
    from kagglehub import KaggleDatasetAdapter
    
    dataset = "abedkhooli/arabic-100k-reviews"
    file_path = ""  # TODO: set the file inside the dataset
    
    df = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        dataset,
        file_path,
    )
df

### Option B: Kaggle Datasets mount (recommended on Kaggle)

When you "Add data" in Kaggle, it appears under `/kaggle/input/<dataset-name>/...`.
Update `DATA_PATH` to point to the CSV.


In [ ]:
if df is None:
    DATA_PATH = "/kaggle/input/arabic-100k-reviews/<FILE.csv>"  # TODO
    if os.path.exists(DATA_PATH):
        df = pd.read_csv(DATA_PATH)
    else:
        print("Set DATA_PATH to your CSV path.")
df.head() if df is not None else None

## 2) Simple EDA

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.isna().mean().sort_values(ascending=False).head(30)

### Guess text + label columns
This is a robust heuristic so the notebook works even if column names differ.

In [ ]:
def guess_text_column(frame: pd.DataFrame) -> str:
    candidates = [
        "review", "text", "content", "comment", "sentence",
        "Review", "Text", "Content"
    ]
    for c in candidates:
        if c in frame.columns:
            return c
    object_cols = [c for c in frame.columns if frame[c].dtype == "object"]
    if not object_cols:
        raise ValueError("No object/text columns found")
    return max(object_cols, key=lambda c: frame[c].fillna("").astype(str).map(len).mean())

def guess_label_column(frame: pd.DataFrame) -> str:
    candidates = ["label", "sentiment", "polarity", "rating", "class", "Label", "Sentiment"]
    for c in candidates:
        if c in frame.columns:
            return c
    # fallback: low-cardinality non-text column
    for c in frame.columns:
        if frame[c].dtype != "object" and frame[c].nunique(dropna=True) <= 10:
            return c
    # last resort: second best object column
    object_cols = [c for c in frame.columns if frame[c].dtype == "object"]
    if len(object_cols) >= 2:
        return object_cols[0]
    return ""

TEXT_COL = guess_text_column(df)
LABEL_COL = guess_label_column(df)
TEXT_COL, LABEL_COL

In [ ]:
df[[TEXT_COL]].head(10)

In [ ]:
if LABEL_COL and LABEL_COL in df.columns:
    display(df[LABEL_COL].value_counts(dropna=False).head(30))
    ax = df[LABEL_COL].value_counts().plot(kind='bar', title=f"Label distribution: {LABEL_COL}")
    ax.set_xlabel("label")
    ax.set_ylabel("count")
    plt.show()
else:
    print("No label column detected; set LABEL_COL manually.")

### Text length stats

In [ ]:
texts = df[TEXT_COL].fillna("").astype(str)
char_len = texts.map(len)
word_len = texts.map(lambda t: len(t.split()))

pd.DataFrame({
    "char_len": char_len.describe(percentiles=[.5,.75,.9,.95,.99]),
    "word_len": word_len.describe(percentiles=[.5,.75,.9,.95,.99]),
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(char_len.clip(0, char_len.quantile(0.99)), bins=60)
axes[0].set_title("Char length (clipped @ p99)")
axes[1].hist(word_len.clip(0, word_len.quantile(0.99)), bins=60)
axes[1].set_title("Word length (clipped @ p99)")
plt.show()

## 3) Semantic EDA (lightweight)

We approximate semantics using:
- **TF-IDF** on Arabic text (word + char ngrams)
- **TruncatedSVD** to 2D (for visualization)
- **KMeans** clustering

This avoids downloading large embedding models and is still useful to spot topical structure.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans

def basic_normalize_ar(text: str) -> str:
    text = str(text)
    # Remove tatweel + diacritics (quick)
    text = re.sub(r"\u0640", "", text)
    text = re.sub(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]", "", text)
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا").replace("ى", "ي").replace("ة", "ه")
    text = re.sub(r"[^\w\u0600-\u06FF]+", " ", text, flags=re.UNICODE)
    return " ".join(text.split()).strip().lower()

sample_n = min(20000, len(df))
sample_df = df.sample(sample_n, random_state=42) if len(df) > sample_n else df.copy()

corpus = sample_df[TEXT_COL].fillna("").astype(str).map(basic_normalize_ar).tolist()

vectorizer = TfidfVectorizer(
    max_features=200000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9,
)
X_word = vectorizer.fit_transform(corpus)

# add char ngrams to capture morphology/spelling variants
vectorizer_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_df=0.9,
    max_features=200000,
)
X_char = vectorizer_char.fit_transform(corpus)

from scipy.sparse import hstack
X = hstack([X_word, X_char]).tocsr()
X.shape

In [ ]:
svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X)
svd.explained_variance_ratio_.sum()

In [ ]:
k = 8
kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
clusters = kmeans.fit_predict(X)

plt.figure(figsize=(8, 6))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=clusters, s=6, alpha=0.5)
plt.title("Semantic-ish map (TF-IDF + SVD) colored by KMeans cluster")
plt.xlabel("svd_1")
plt.ylabel("svd_2")
plt.show()


### Inspect clusters: top terms + examples
We use word-TF-IDF terms for readability.

In [ ]:
terms = np.array(vectorizer.get_feature_names_out())
centroids = kmeans.cluster_centers_[:, :X_word.shape[1]]  # only word part

def top_terms_for_cluster(ci: int, n: int = 15):
    top_idx = np.argsort(centroids[ci])[-n:][::-1]
    return terms[top_idx].tolist()

for ci in range(k):
    print(f"\nCluster {ci} — top terms:")
    print(", ".join(top_terms_for_cluster(ci, 18)))
    ex = sample_df.loc[np.where(clusters == ci)[0], [TEXT_COL]].head(3)
    display(ex)


## 4) (Optional) Weak aspect labeling

If you clone this repo into the notebook environment, you can run the weak labeler
to create aspect columns (`service/logistics/location/cleaning`).

In this repo: `absa_aspect_labeling.py`.


In [ ]:
# Example (if the repo files are present in the notebook runtime):
# !python3 ../absa_aspect_labeling.py --csv /kaggle/input/arabic-100k-reviews/<FILE.csv> --out labeled_reviews.csv
None